In [1]:
# prompt: 구글드라이브연결

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### 1. 카테고리 분류

In [ ]:
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# CategoryClassifier 클래스 정의
class CategoryClassifier(nn.Module):
    def __init__(self, emb_dim=1024, hidden_dim=512):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, q_emb_batch, cat_emb_all):
        batch_size = q_emb_batch.size(0)
        num_categories = cat_emb_all.size(0)
        q_emb_batch = q_emb_batch.unsqueeze(1)
        q_emb_batch = q_emb_batch.expand(batch_size, num_categories, q_emb_batch.size(-1))
        cat_emb_all = cat_emb_all.unsqueeze(0)
        cat_emb_all = cat_emb_all.expand(batch_size, num_categories, cat_emb_all.size(-1))
        x = torch.cat([q_emb_batch, cat_emb_all], dim=-1)
        x = x.view(batch_size * num_categories, -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = x.view(batch_size, num_categories)
        return x

In [ ]:
# CategoryClassifier랑 카테고리 임베딩 벡터 로드
# model_path : CategoryClassifier와 카테고리 임베딩 벡터의 가중치가 있는 폴더의 경로
# device : torch gpu device
def load_classifer(model_path, device):
    classifier = CategoryClassifier(emb_dim=1024, hidden_dim=512)
    classifier.load_state_dict(torch.load(model_path + '/category_classifier.pth'))
    classifier.to(device)

    # NumPy 배열을 파일에서 불러오기
    category_embeddings_np = np.load(model_path + '/category_embeddings.npy')

    # NumPy 배열을 PyTorch 텐서로 변환
    category_embeddings = torch.from_numpy(category_embeddings_np).to(device)

    return classifier, category_embeddings

In [ ]:
# 카테고리 분류 하는 함수
# classifier : 위에서 로드한 CategoryClassifier
# category_embeddings : 위에서 로드한 카테고리 임베딩 벡터
# embedding_model : 미리 로드해둔 "intfloat/multilingual-e5-large-instruct"
# device : torch gpu device
def classify_category(classifier, category_embeddings, embedding_model, device, question: str):
    # 54개 카테고리
    CATEGORIES = ['LPG충전소', 'OTT', '간편결제', '게임', '공과금', '공연', '기업형슈퍼마켓', '대중교통',
          '대형마트', '드럭스토어', '디지털구독', '렌탈', '리조트', '멤버십구독', '면세점', '미용', '미용실',
          '반려동물', '배달', '백화점', '보험', '세탁소', '쇼핑', '스포츠', '아울렛', '여행', '영화관',
          '온라인서점', '온라인쇼핑', '우체국', '웹툰', '음식점', '의료', '전기차충전소', '제과아이스크림',
          '주유소', '주차장', '차량정비소', '창고형할인매장', '카페', '콘도', '타이어샵', '택시', '테마파크',
          '통신사', '패션', '편의점', '학습지', '학원', '항공사', '해외대중교통', '호텔', '호텔음식점',
          '홈쇼핑']

    classifier.eval()
    with torch.no_grad():
        # (1, 1024)
        q_emb = embedding_model.encode(question, convert_to_tensor=True).unsqueeze(0).to(device)

        logits = classifier(q_emb, category_embeddings)  # (1, 54)
        probs = torch.softmax(logits, dim=1)             # 확률 분포 (1, 54)

        pred_idx = torch.argmax(probs, dim=1).item()     # 0~53
        return CATEGORIES[pred_idx]

In [ ]:
# 임베딩 모델 load
embedding_model = SentenceTransformer("intfloat/multilingual-e5-large-instruct").to(device)

# CategoryClassifier랑 카테고리 임베딩 벡터 로드
classifier, category_embeddings = load_classifer('/content/drive/MyDrive/Project_final/model', device)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

<ipython-input-4-09b36c9ae740>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  classifier.load_state_dict(torch.load(model_path + '/category_classifier.pth'))


In [ ]:
test_questions = [
    "스타벅스에 자주가는데 할인이 많이 되는 카드를 알려주세요",
    "교통비 할인이 잘되는 카드가 있나요?",
    "인터넷에서 쇼핑을 자주하는데 할인율이 높은 카드를 알려주세요.",
    "카페랑 음식점에서 할인이 잘되는 카드를 알려주세요.",
    "쇼핑할 때 할인 많이 받을 수 있는 카드 추천해줘.",
    "여행 다닐 때 혜택 좋은 카드가 뭐야?",
    "주유소에서 할인 많이 되는 카드 알려줘.",
    "음식점에서 사용하기 좋은 카드 추천해줄래?",
    "영화 할인 혜택 있는 카드 있어?",
    "항공 마일리지 적립 잘 되는 카드 찾고 있어.",
    "편의점에서 쓸만한 카드 뭐야?",
    "교통비 할인 혜택이 있는 카드 추천 부탁해.",
    "외식할 때 사용하기 좋은 카드는 뭐야?",
    "넷플릭스 구독비 할인되는 카드를 추천해주세요.",
    "멜론 월정액 할인되는 카드가 뭔가요",
    "호텔 뷔페를 자주가는데 좋은 카드를 추천해주세요.",
    "호텔에 자주가는데 좋은 카드를 추천해주세요.",
    "쿠팡 멤버쉽 구독비 할인 잘되는 카드가 뭔가요",
    "네이버플러스 멤버십 구독비 할인 되는 카드를 알려주세요",
    "애완동물을 키우는데 좋은 카드를 추천해주세요.",
    "개를 키우는데 좋은 카드가 있을까요?",
    "LPG를 연료로 하는 차를 타고다닙니다. 좋은 카드를 추천해주세요.",
    "LPG 충전할 때 할인이 많이 되는 카드를 추천해주세요.",
    "다음달에 싱가폴에 가는데 좋은 카드를 골라주세요",
    "밖에서 밥을 자주 사먹는데 사용하기 좋은 카드를 골라줘",
    "쿠팡와우 구독비 줄이고싶은데 좋은 카드가 있을까요"
]

q_c_list = []

for q in test_questions:
    pred_cat = classify_category(classifier, category_embeddings, embedding_model, device, q)
    print(f"질문: {q}")
    print(f"=> 예측 카테고리: {pred_cat}")
    print("----")
    q_c_list.append((q, pred_cat))

질문: 스타벅스에 자주가는데 할인이 많이 되는 카드를 알려주세요
=> 예측 카테고리: 카페
----
질문: 교통비 할인이 잘되는 카드가 있나요?
=> 예측 카테고리: 대중교통
----
질문: 인터넷에서 쇼핑을 자주하는데 할인율이 높은 카드를 알려주세요.
=> 예측 카테고리: 온라인쇼핑
----
질문: 카페랑 음식점에서 할인이 잘되는 카드를 알려주세요.
=> 예측 카테고리: 음식점
----
질문: 쇼핑할 때 할인 많이 받을 수 있는 카드 추천해줘.
=> 예측 카테고리: 쇼핑
----
질문: 여행 다닐 때 혜택 좋은 카드가 뭐야?
=> 예측 카테고리: 여행
----
질문: 주유소에서 할인 많이 되는 카드 알려줘.
=> 예측 카테고리: 주유소
----
질문: 음식점에서 사용하기 좋은 카드 추천해줄래?
=> 예측 카테고리: 음식점
----
질문: 영화 할인 혜택 있는 카드 있어?
=> 예측 카테고리: 영화관
----
질문: 항공 마일리지 적립 잘 되는 카드 찾고 있어.
=> 예측 카테고리: 항공사
----
질문: 편의점에서 쓸만한 카드 뭐야?
=> 예측 카테고리: 편의점
----
질문: 교통비 할인 혜택이 있는 카드 추천 부탁해.
=> 예측 카테고리: 대중교통
----
질문: 외식할 때 사용하기 좋은 카드는 뭐야?
=> 예측 카테고리: 음식점
----
질문: 넷플릭스 구독비 할인되는 카드를 추천해주세요.
=> 예측 카테고리: OTT
----
질문: 멜론 월정액 할인되는 카드가 뭔가요
=> 예측 카테고리: 디지털구독
----
질문: 호텔 뷔페를 자주가는데 좋은 카드를 추천해주세요.
=> 예측 카테고리: 호텔음식점
----
질문: 호텔에 자주가는데 좋은 카드를 추천해주세요.
=> 예측 카테고리: 호텔
----
질문: 쿠팡 멤버쉽 구독비 할인 잘되는 카드가 뭔가요
=> 예측 카테고리: 멤버십구독
----
질문: 네이버플러스 멤버십 구독비 할인 되는 카드를 알려주세요
=> 예측 카테고리: 멤버십구독
----
질문: 애완동물을 키우는데 좋은 카드를 추천해주세요.
=> 예측 카

In [ ]:
print(q_c_list)

[('스타벅스에 자주가는데 할인이 많이 되는 카드를 알려주세요', '카페'), ('교통비 할인이 잘되는 카드가 있나요?', '대중교통'), ('인터넷에서 쇼핑을 자주하는데 할인율이 높은 카드를 알려주세요.', '온라인쇼핑'), ('카페랑 음식점에서 할인이 잘되는 카드를 알려주세요.', '음식점'), ('쇼핑할 때 할인 많이 받을 수 있는 카드 추천해줘.', '쇼핑'), ('여행 다닐 때 혜택 좋은 카드가 뭐야?', '여행'), ('주유소에서 할인 많이 되는 카드 알려줘.', '주유소'), ('음식점에서 사용하기 좋은 카드 추천해줄래?', '음식점'), ('영화 할인 혜택 있는 카드 있어?', '영화관'), ('항공 마일리지 적립 잘 되는 카드 찾고 있어.', '항공사'), ('편의점에서 쓸만한 카드 뭐야?', '편의점'), ('교통비 할인 혜택이 있는 카드 추천 부탁해.', '대중교통'), ('외식할 때 사용하기 좋은 카드는 뭐야?', '음식점'), ('넷플릭스 구독비 할인되는 카드를 추천해주세요.', 'OTT'), ('멜론 월정액 할인되는 카드가 뭔가요', '디지털구독'), ('호텔 뷔페를 자주가는데 좋은 카드를 추천해주세요.', '호텔음식점'), ('호텔에 자주가는데 좋은 카드를 추천해주세요.', '호텔'), ('쿠팡 멤버쉽 구독비 할인 잘되는 카드가 뭔가요', '멤버십구독'), ('네이버플러스 멤버십 구독비 할인 되는 카드를 알려주세요', '멤버십구독'), ('애완동물을 키우는데 좋은 카드를 추천해주세요.', '반려동물'), ('개를 키우는데 좋은 카드가 있을까요?', '반려동물'), ('LPG를 연료로 하는 차를 타고다닙니다. 좋은 카드를 추천해주세요.', 'LPG충전소'), ('LPG 충전할 때 할인이 많이 되는 카드를 추천해주세요.', 'LPG충전소'), ('다음달에 싱가폴에 가는데 좋은 카드를 골라주세요', '여행'), ('밖에서 밥을 자주 사먹는데 사용하기 좋은 카드를 골라줘', '음식점'), ('쿠팡와우 구독비 줄이고싶은데 좋은 카드가 있을까요

### 2. 상호명 추출

In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.0 MB/s eta 0:00:00


In [ ]:
####### 최종 ########
import pandas as pd
import json
from rapidfuzz import fuzz
from rapidfuzz import process
import re


# JSON 파일을 딕셔너리로 로드하는 함수
def load_json_to_dict(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
        print("JSON 파일이 성공적으로 로드되었습니다.")
        return data
    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다: {file_path}")
    except json.JSONDecodeError as e:
        print(f"JSON 디코딩 에러: {e}")
    except Exception as e:
        print(f"알 수 없는 오류 발생: {e}")


# JSON 파일 불러오기
file_path = '/content/drive/MyDrive/Project_final/data/target_list_for_matching.json'  # 업종, 상호명 있는 json 파일
json_dict = load_json_to_dict(file_path)


# 정규 표현식을 사용하여 문장에 알파벳이 있는지 찾고 대문자로 변환
def capitalize_english_words(text):
    return re.sub(r'\b[a-zA-Z]+\b', lambda match: match.group(0).upper(), text)


# 질문에 상호명이 있으면 상호명을 반환, 없으면 업종명을 다시 반환
# 함수호출 매개변수 : question:질문, category:업종, target_list:업종,상호명 json 파일을 변환한 dictionary 변수
def extract_store(question, category, target_list):
    cut_off_score = 90 # 업종, 상호명 출력 rapidfuzz 점수 기준
    temp_max = 0 # rapidfuzz 점수 최대값 판별용 변수
    result = '' # 상호명을 출력할 경우 rapidfuzz 점수가 최대 값인 상호명
    question = question.replace(' ', '') # 질문의 띄어쓰기 제거
    question = capitalize_english_words(question) # 질문에 있는 알파벳 모두 대문자로

    if len(target_list[category]) != 0: # 업종 하위에 상호명이 있을 경우 상호명 탐색
        # 긴 단어부터 우선순위 설정
        keywords = sorted(target_list[category], key=len, reverse=True)

        for store in keywords:
            store_original = store
            if '(LPG)' in store: store = store.split('(LPG)')[0]

            score = fuzz.partial_ratio(store, question) # question과 상호명으로 rapidfuzz 점수 계산

            if temp_max < score: # rapidfuzz 점수가 제일 높은 상호명 판별
                temp_max = score
                result = store_original

        if temp_max >= cut_off_score: # 기준치 이상이면 특정 상호면 리턴
            return result#, str(temp_max)

        else: # 기준치 이하일때 업종명 리턴
            return category#, result + '/' + str(temp_max)

    else: # 업종 하위에 상호명이 없을 경우 업종명 리턴
        return category#, '100'

JSON 파일이 성공적으로 로드되었습니다.


In [ ]:
list_incorrect = []

# 각 row를 하나씩 처리
for question, category in q_c_list:
    result = extract_store(question, category, json_dict)
    list_incorrect.append([question, result])

for i in list_incorrect:
    print(i)

['스타벅스에 자주가는데 할인이 많이 되는 카드를 알려주세요', '스타벅스']
['교통비 할인이 잘되는 카드가 있나요?', '대중교통']
['인터넷에서 쇼핑을 자주하는데 할인율이 높은 카드를 알려주세요.', '온라인쇼핑']
['카페랑 음식점에서 할인이 잘되는 카드를 알려주세요.', '음식점']
['쇼핑할 때 할인 많이 받을 수 있는 카드 추천해줘.', '쇼핑']
['여행 다닐 때 혜택 좋은 카드가 뭐야?', '여행']
['주유소에서 할인 많이 되는 카드 알려줘.', '주유소']
['음식점에서 사용하기 좋은 카드 추천해줄래?', '음식점']
['영화 할인 혜택 있는 카드 있어?', '영화관']
['항공 마일리지 적립 잘 되는 카드 찾고 있어.', '항공사']
['편의점에서 쓸만한 카드 뭐야?', '편의점']
['교통비 할인 혜택이 있는 카드 추천 부탁해.', '대중교통']
['외식할 때 사용하기 좋은 카드는 뭐야?', '음식점']
['넷플릭스 구독비 할인되는 카드를 추천해주세요.', '넷플릭스']
['멜론 월정액 할인되는 카드가 뭔가요', '멜론']
['호텔 뷔페를 자주가는데 좋은 카드를 추천해주세요.', '호텔음식점']
['호텔에 자주가는데 좋은 카드를 추천해주세요.', '호텔']
['쿠팡 멤버쉽 구독비 할인 잘되는 카드가 뭔가요', '멤버십구독']
['네이버플러스 멤버십 구독비 할인 되는 카드를 알려주세요', '네이버플러스']
['애완동물을 키우는데 좋은 카드를 추천해주세요.', '반려동물']
['개를 키우는데 좋은 카드가 있을까요?', '반려동물']
['LPG를 연료로 하는 차를 타고다닙니다. 좋은 카드를 추천해주세요.', 'LPG충전소']
['LPG 충전할 때 할인이 많이 되는 카드를 추천해주세요.', 'LPG충전소']
['다음달에 싱가폴에 가는데 좋은 카드를 골라주세요', '여행']
['밖에서 밥을 자주 사먹는데 사용하기 좋은 카드를 골라줘', '음식점']
['쿠팡와우 구독비 줄이고싶은데 좋은 카드가 있을까요', '멤버십구독']
